# ارزیابی و آماده‌سازی داده‌های آزمون

این نوت‌بوک نقش «داده و ارزیابی» در گام ۲ پروژه تشخیص ارقام دست‌نویس را انجام می‌دهد: بررسی کیفیت، پاک‌سازی قابل بازتولید، جداسازی اعتبارسنجی و حفظ آزمون مستقل. **مدلی در این مرحله آموزش یا انتخاب نمی‌شود.**

## منبع و اجرای نوت‌بوک

داده ورودی فایل `handwritten_digits_data.npz` همین پوشه است که توسط `g2.ipynb` از مجموعه [MagedSaeed/MADBase](https://huggingface.co/datasets/MagedSaeed/MADBase) تهیه شده است. فایل شامل تصاویر خاکستری ۲۸×۲۸ و برچسب‌های ۰ تا ۹ در دو بخش اصلی `train` و `test` است. برای اطلاعات بیشتر و مجوز به صفحه منبع مراجعه کنید؛ وضعیت مجوز از روی فایل محلی قابل تأیید نیست.

در Jupyter یا Colab همه سلول‌ها را به ترتیب اجرا کنید. پیش‌نیازها: Python، NumPy، Matplotlib و scikit-learn. در صورت نیاز: `%pip install numpy matplotlib scikit-learn`. فایل داده باید کنار نوت‌بوک یا در مسیر `step-2` نسبت به پوشه فعلی باشد. بارگذاری فایل به شبکه نیاز ندارد.


In [ ]:
from pathlib import Path
from collections import defaultdict
import sys
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
candidates = [Path("handwritten_digits_data.npz"), Path("step-2/handwritten_digits_data.npz")]
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("فایل handwritten_digits_data.npz را کنار نوت‌بوک یا در step-2 قرار دهید.")
with np.load(DATA_PATH, allow_pickle=False) as data:
    required = {"train_images", "train_labels", "test_images", "test_labels"}
    if not required.issubset(data.files):
        raise ValueError(f"کلیدهای مورد نیاز یافت نشد: {sorted(required - set(data.files))}")
    train_images = data["train_images"]
    train_labels = data["train_labels"]
    test_images = data["test_images"]
    test_labels = data["test_labels"]
print("مسیر داده:", DATA_PATH.resolve())
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("بذر تقسیم:", SEED)


## ۱. بررسی ساختار و کیفیت

بررسی‌های خودکار شامل ابعاد، نوع و بازه پیکسل، برچسب‌های معتبر و تصاویر کاملاً خالی است. تصویر خالی یا دارای یک مقدار ثابت فقط گزارش می‌شود، چون حذف خودکار آن بدون بررسی انسانی می‌تواند نمونه معتبر را حذف کند. این بررسی جای بازبینی بصری یا تأیید درستی برچسب‌ها را نمی‌گیرد.


In [ ]:
def inspect_split(name, images, labels):
    if images.ndim != 3 or images.shape[1:] != (28, 28):
        raise ValueError(f"{name}: انتظار آرایه تصاویر با شکل (N, 28, 28) می‌رود؛ دریافت شد {images.shape}")
    if labels.ndim != 1 or len(images) != len(labels) or len(labels) == 0:
        raise ValueError(f"{name}: تعداد یا شکل تصاویر و برچسب‌ها معتبر نیست.")
    finite_images = np.isfinite(images).all(axis=(1, 2))
    in_range = ((images >= 0) & (images <= 255)).all(axis=(1, 2))
    valid_labels = np.isin(labels, np.arange(10))
    valid = finite_images & in_range & valid_labels
    flat = images.reshape(len(images), -1)
    print(f"\n{name}: تصاویر {images.shape} ({images.dtype})؛ برچسب‌ها {labels.shape} ({labels.dtype})")
    print("بازه پیکسل:", np.nanmin(images), "تا", np.nanmax(images))
    print("برچسب‌ها و تعداد:", dict(zip(*np.unique(labels, return_counts=True))))
    print("پیکسل غیرمتناهی:", int((~finite_images).sum()), "| خارج از بازه:", int((~in_range).sum()), "| برچسب نامعتبر:", int((~valid_labels).sum()))
    print("تصویر تماماً صفر:", int((flat == 0).all(axis=1).sum()), "| تصویر با شدت ثابت:", int((flat == flat[:, :1]).all(axis=1).sum()))
    print("کل ردیف‌های نامعتبر (بدون شمارش تکراری):", int((~valid).sum()))
    return valid

train_valid = inspect_split("train اصلی", train_images, train_labels)
test_valid = inspect_split("test اصلی", test_images, test_labels)


## ۲. پاک‌سازی و بررسی هم‌پوشانی

ردیف نامعتبر حذف می‌شود و شناسه آن (شماره ردیف فایل اصلی، از صفر) نگهداری می‌شود. مجموعه آزمون اصلی حفظ می‌شود، مگر ردیف نامعتبر؛ تصویر تکراری یا برچسب متناقض درون آن فقط گزارش می‌شود تا آزمون رسمی بی‌دلیل تغییر نکند. برای جلوگیری از نشت، تمام تصاویر آموزش که عیناً در آزمون هستند حذف می‌شوند، حتی اگر برچسب متفاوت باشد. در باقیمانده آموزش، تصاویر عیناً یکسان با برچسب سازگار به یک ردیف کاهش می‌یابند و همه نسخه‌های دارای برچسب متناقض کنار گذاشته می‌شوند. بررسی صرفاً برای تصاویر **دقیقاً** یکسان است؛ تصاویر مشابه یا نویسندگان مشترک ممکن است همچنان وجود داشته باشند.


In [ ]:
# کلید، خود پیکسل‌های تصویر است؛ نه برچسب و نه هش احتمالیِ برخورددار.
def image_key(image):
    return image.tobytes()

test_ids = np.flatnonzero(test_valid)
test_groups = defaultdict(list)
for i in test_ids:
    test_groups[image_key(test_images[i])].append(i)
test_duplicate_extras = sum(len(group) - 1 for group in test_groups.values())
test_conflicts = [group for group in test_groups.values() if len(np.unique(test_labels[group])) > 1]
print("نسخه‌های تکراری در آزمون (حفظ شده):", test_duplicate_extras)
print("گروه‌های آزمون با تصویر یکسان و برچسب متناقض (نیازمند بررسی):", len(test_conflicts))

train_groups = defaultdict(list)
overlap_ids = []
overlap_label_conflicts = 0
for i in np.flatnonzero(train_valid):
    key = image_key(train_images[i])
    if key in test_groups:
        overlap_ids.append(i)
        if np.any(test_labels[test_groups[key]] != train_labels[i]):
            overlap_label_conflicts += 1
    else:
        train_groups[key].append(i)

clean_train_ids, duplicate_ids, conflict_ids = [], [], []
for group in train_groups.values():
    if len(np.unique(train_labels[group])) > 1:
        conflict_ids.extend(group)
    else:
        clean_train_ids.append(group[0])
        duplicate_ids.extend(group[1:])
clean_train_ids = np.array(sorted(clean_train_ids), dtype=int)
print("ردیف آموزش هم‌پوشان با آزمون، حذف‌شده:", len(overlap_ids))
print("از آن‌ها با حداقل یک برچسب آزمون متفاوت:", overlap_label_conflicts)
print("نسخه‌های تکراری آموزش حذف‌شده:", len(duplicate_ids))
print("ردیف‌های آموزش با برچسب متناقض حذف‌شده:", len(conflict_ids))
print("آموزش پاک‌شده:", len(clean_train_ids), "| آزمون معتبر:", len(test_ids))
assert len(clean_train_ids) + len(duplicate_ids) + len(conflict_ids) + len(overlap_ids) == int(train_valid.sum())


## ۳. جداسازی اعتبارسنجی از آموزش

بخش آزمون از قبل در فایل منبع جداست و دوباره با آموزش مخلوط نمی‌شود. از آموزش پاک‌شده، ۲۰٪ به روش طبقه‌بندی‌شده و با بذر ثابت برای اعتبارسنجی جدا می‌کنیم. مقادیر تصاویر به صورت `uint8` در بازه ۰ تا ۲۵۵ باقی می‌مانند. در زمان مدل‌سازی می‌توان هر سه بخش را به یک روش ثابت بر ۲۵۵ تقسیم کرد؛ هر تبدیل یادگرفتنی باید فقط روی بخش آموزش برازش شود.


In [ ]:
y_clean = train_labels[clean_train_ids]
ids_train, ids_val = train_test_split(clean_train_ids, test_size=0.2, random_state=SEED, stratify=y_clean)
X_train, y_train = train_images[ids_train], train_labels[ids_train]
X_val, y_val = train_images[ids_val], train_labels[ids_val]
X_test, y_test = test_images[test_ids], test_labels[test_ids]
assert not (set(ids_train) & set(ids_val))
assert len(ids_train) + len(ids_val) == len(clean_train_ids)
train_keys = {image_key(image) for image in X_train}
val_keys = {image_key(image) for image in X_val}
assert not (train_keys & val_keys or train_keys & test_groups.keys() or val_keys & test_groups.keys())
print("کنترل موفق: هیچ تصویر دقیقاً یکسانی میان آموزش، اعتبارسنجی و آزمون نیست.")


## ۴. گزارش مجموعه‌ها و نمونه‌های آزمون

جدول تعداد هر رقم و نمودار توزیع آزمون برای بررسی عدم توازن است. چند تصویر آزمون صرفاً برای کنترل کیفیت نمایش داده می‌شوند؛ از آن‌ها برای تنظیم مدل استفاده نکنید.


In [ ]:
print(f"{'بخش':<12} {'تعداد':>7}   تعداد هر رقم از ۰ تا ۹")
for name, y in (("train", y_train), ("validation", y_val), ("test", y_test)):
    print(f"{name:<12} {len(y):>7}   {np.bincount(y.astype(int), minlength=10).tolist()}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(10), np.bincount(y_test.astype(int), minlength=10))
ax.set(xlabel="Digit", ylabel="Count", title="Test set class distribution", xticks=np.arange(10))
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for digit, ax in enumerate(axes):
    matches = np.flatnonzero(y_test == digit)
    if len(matches):
        ax.imshow(X_test[matches[0]], cmap="gray", vmin=0, vmax=255)
    ax.set_title(str(digit))
    ax.axis("off")
plt.tight_layout()
plt.show()


## ۵. ذخیره اختیاری خروجی

برای تولید `madbase_clean_splits.npz` در پوشه جاری، `SAVE_SPLITS` را `True` کنید. فایل تولیدشده، تصاویر، برچسب‌ها، شناسه ردیف‌های اصلی و بذر تقسیم را دارد. شناسه‌های آموزش و اعتبارسنجی به `train` اصلی و شناسه آزمون به `test` اصلی ارجاع می‌دهند. از ثبت فایل تولیدی بزرگ در مخزن بدون هماهنگی خودداری کنید؛ داده و کد امکان بازتولید خروجی را فراهم می‌کنند.


In [ ]:
SAVE_SPLITS = False
if SAVE_SPLITS:
    np.savez_compressed("madbase_clean_splits.npz", X_train=X_train, y_train=y_train,
                        X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test,
                        ids_train=ids_train, ids_val=ids_val, ids_test=test_ids,
                        seed=SEED, sklearn_version=sklearn.__version__)
    print("فایل madbase_clean_splits.npz ذخیره شد.")
else:
    print("ذخیره غیرفعال است؛ برای ساخت فایل SAVE_SPLITS را True کنید.")


## ۶. ارزیابی اختیاری مدل نهایی

وقتی مدل فقط با آموزش و اعتبارسنجی انتخاب و تثبیت شد، پیش‌بینی آن روی `X_test` را **به همان ترتیب** در `y_test_pred` قرار دهید و سلول زیر را اجرا کنید. بدون پیش‌بینی، هیچ نتیجه‌ای ساخته نمی‌شود. استفاده از نتیجه آزمون برای تنظیم مجدد مدل باعث نشت اطلاعات می‌شود. طبق معیارهای `step-1/notes/risks.md`، دقت، Precision، Recall، F1 و ماتریس درهم‌ریختگی گزارش می‌شوند.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

if "y_test_pred" not in globals():
    print("پیش‌بینی مدل نهایی ارائه نشده؛ ارزیابی مدل انجام نشد.")
else:
    y_pred = np.asarray(y_test_pred)
    if y_pred.shape != y_test.shape or not np.isin(y_pred, np.arange(10)).all():
        raise ValueError("y_test_pred باید به ترتیب آزمون، برای هر تصویر یک برچسب ۰ تا ۹ داشته باشد.")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, labels=np.arange(10), zero_division=0))
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(10))
    ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(cmap="Blues")
    plt.show()
